# ML-1 V4 · Независимая проверка 870 660 прогнозов
### Пострасчётный научный аудит, проверка воспроизводимости и таблицы для публикации

**Объект аудита:** опубликованный и неизменяемый релиз `stage2-ml1-v4.0.0-20261026T130048Z-58bd8971` приватного репозитория `AntonDozhdikov/Demograf_sovetnik`.

Ноутбук самостоятельно скачивает из GitHub Release `ML1_V4_PREDICTIONS.xlsx` (около 78 МБ) и `ML1_V4_RESULTS.xlsx`, сверяет оба SHA-256 с `run_manifest.json`, закреплённым за коммитом релиза. **Перевычисления моделей, изменения GitHub и доступ к исходной базе не производятся.**

Проверяются три горизонта (1/3/5 года), четыре протокола проверки, все модели и seeds, абсолютные потоки, бинарные вероятности и интервалы. Отдельно приводится новый **эксплоративный** анализ добавочной роли факторов при **одинаковом алгоритме и идентичном TEST row_id**; это не заменяет пререгистрированные гипотезы исходного эксперимента.

Результат — ровно **две Excel-книги и один ZIP с десятью PNG-графиками**.

## 0. Среда и авторизация

В Google Colab открыть **Secrets** и добавить `GITHUB_TOKEN` с доступом к закрытому репозиторию (Contents: Read; Release assets: Read). Значение токена не выводится. `Runtime → Run all`; GPU не нужен. Исходный архив Predictions оставлять не нужно загружать вручную.

In [1]:
# Установка до любых тестов (включая XlsxWriter, ранее отсутствовавший в окружении).
%pip -q install "numpy>=1.26" "pandas>=2.2" "scipy>=1.12" "openpyxl>=3.1" "XlsxWriter>=3.2" "matplotlib>=3.8" "scikit-learn>=1.4" "requests>=2.31" "urllib3>=2" "tqdm>=4.66"

In [2]:
# 01 — Parameters, real private GitHub Release, SHA256-checked download
import os, json, hashlib, time, zipfile, warnings, math, tempfile, shutil
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm
from openpyxl import load_workbook
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPO='AntonDozhdikov/Demograf_sovetnik'
TAG='stage2-ml1-v4.0.0-20261026T130048Z-58bd8971'
MANIFEST_PATH='stage2/ml01_migration/run_manifest.json'
PRED_NAME='ML1_V4_PREDICTIONS.xlsx'
RESULTS_NAME='ML1_V4_RESULTS.xlsx'
BASE_DIR=Path('/content/ML1_V4_PREDICTION_AUDIT' if Path('/content').exists() else '/mnt/data/ML1_V4_PREDICTION_AUDIT')
INPUT_DIR=BASE_DIR/'input'; OUTPUT_DIR=BASE_DIR/'publication'; PLOTS_DIR=BASE_DIR/'plots'
for d in (INPUT_DIR,OUTPUT_DIR,PLOTS_DIR):d.mkdir(parents=True,exist_ok=True)
# For CPU-efficient reproducible region-bootstrap, not the original V4 bootstrap.
BOOTSTRAPS=1500; BOOT_SEED=20261027
PAIR_SCOPE='ALL_SAME_MODEL_FAMILY_TRANSITIONS'
FIGURE_DPI=175

def sha256(path,chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''):h.update(b)
    return h.hexdigest()

def get_token():
    token=os.environ.get('GITHUB_TOKEN','').strip() or os.environ.get('GH_TOKEN','').strip()
    if not token:
        try:
            from google.colab import userdata
            token=(userdata.get('GITHUB_TOKEN') or '').strip()
        except Exception: pass
    if not token:raise RuntimeError('Нет GITHUB_TOKEN. Colab → Secrets → GITHUB_TOKEN; private Release не доступен анонимно.')
    return token

def session_for(token):
    sess=requests.Session()
    sess.headers.update({'Authorization':f'Bearer {token}',
        'X-GitHub-Api-Version':'2022-11-28','User-Agent':'ML1-V4-Predictions-Independent-Audit/1.0'})
    retry=Retry(total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504],allowed_methods=['GET'])
    sess.mount('https://',HTTPAdapter(max_retries=retry))
    return sess

def gh_json(sess,url):
    r=sess.get(url,timeout=(15,65),headers={'Accept':'application/vnd.github+json'})
    if r.status_code in (401,403,404):
        raise RuntimeError(f'GitHub HTTP {r.status_code}. Проверьте права Contents:Read и Releases:Read для приватного {REPO}; URL={url.split("?")[0]}')
    r.raise_for_status();return r.json()

def github_manifest_and_release(sess):
    api=f'https://api.github.com/repos/{REPO}'
    release=gh_json(sess,f'{api}/releases/tags/{TAG}')
    if release.get('draft') or release.get('prerelease'):
        raise ValueError('Публикация не является зафиксированным штатным Release.')
    ref=gh_json(sess,f'{api}/git/ref/tags/{TAG}')
    commit=ref['object']['sha']
    if ref['object']['type']=='tag':
        obj=gh_json(sess,f'{api}/git/tags/{commit}')
        commit=obj['object']['sha']
    if release.get('target_commitish') and len(release['target_commitish'])==40:
        if release['target_commitish']!=commit:
            raise ValueError('Release target_commitish != tag SHA: отказ от смешения версий')
    blob=gh_json(sess,f'{api}/contents/{MANIFEST_PATH}?ref={commit}')
    import base64
    if blob.get('encoding')!='base64':raise ValueError('Неожиданный формат GitHub manifest')
    manifest=json.loads(base64.b64decode(blob['content']))
    if manifest.get('n_prediction_rows')!=870660:
        raise ValueError('Манифест не соответствует зафиксированному V4 — ожидалось 870660.')
    assets={x['name']:x for x in release['assets']}
    for name in (PRED_NAME,RESULTS_NAME):
        if name not in assets or name not in manifest['outputs']:
            raise FileNotFoundError('Нет подтверждённого Release asset: '+name)
        if assets[name]['size']!=manifest['outputs'][name]['size']:
            raise ValueError('GitHub asset size противоречит manifest: '+name)
    print('Зафиксированная публикация:',release['html_url'])
    print('Git commit:',commit)
    return manifest,assets,commit,release['html_url']

def download_asset(sess,asset,expected_meta,destination):
    destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)
    expected_sha=expected_meta['sha256'];size=int(expected_meta['size'])
    if destination.is_file() and destination.stat().st_size==size and sha256(destination)==expected_sha:
        print('Кэш подтверждён SHA256:',destination.name);return destination
    destination.unlink(missing_ok=True)
    api=f'https://api.github.com/repos/{REPO}/releases/assets/{asset["id"]}'
    temp=destination.with_suffix('.part')
    try:
        with sess.get(api,headers={'Accept':'application/octet-stream'},stream=True,timeout=(20,180),allow_redirects=True) as r:
            r.raise_for_status()
            typ=r.headers.get('Content-Type','').lower()
            if 'json' in typ:raise RuntimeError('GitHub вернул JSON вместо бинарного Release asset: проверьте права токена.')
            digest=hashlib.sha256();count=0
            with open(temp,'wb') as f,tqdm(total=size,unit='B',unit_scale=True,desc=destination.name) as bar:
                for block in r.iter_content(chunk_size=4*1024*1024):
                    if block:
                        f.write(block);digest.update(block);count+=len(block);bar.update(len(block))
        if count!=size or digest.hexdigest()!=expected_sha:
            raise ValueError(f'КОНТРОЛЬНАЯ СУММА НЕ СОВПАЛА: {destination.name} {count}/{size} bytes')
        with open(temp,'rb') as f:
            if f.read(4)!=b'PK\x03\x04':raise ValueError('Release asset не является XLSX (ZIP)')
        if not zipfile.is_zipfile(temp):raise ValueError('Недействительный ZIP-контейнер XLSX')
        temp.replace(destination)
        print('SHA256 PASS:',destination.name,expected_sha)
        return destination
    finally:temp.unlink(missing_ok=True)

In [3]:
# 00A — dependency, XLSX and GitHub binary download regression tests BEFORE network
import io
with tempfile.TemporaryDirectory() as td:
    td=Path(td);sample=td/'fixture.xlsx'
    import xlsxwriter
    w=xlsxwriter.Workbook(str(sample));s=w.add_worksheet('Fixture')
    s.write_row(0,0,['row_id','truth','prediction']);s.write_row(1,0,['id0001',1.5,1.2]);w.close()
    assert zipfile.is_zipfile(sample)
    assert list(load_workbook(sample,read_only=True)['Fixture'].values)[1]==('id0001',1.5,1.2)
    binary=sample.read_bytes();expected_sha=hashlib.sha256(binary).hexdigest()
    class FakeResponse:
        headers={'Content-Type':'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}
        def __init__(self,data):self.data=data
        def __enter__(self):return self
        def __exit__(self,*exc):pass
        def raise_for_status(self):pass
        def iter_content(self,chunk_size):yield self.data
    class FakeSession:
        def __init__(self,data):self.data=data
        def get(self,*a,**kw):return FakeResponse(self.data)
    info={'sha256':expected_sha,'size':len(binary)}
    downloaded=download_asset(FakeSession(binary),{'id':1},info,td/'from_github.xlsx')
    assert downloaded.read_bytes()==binary and sha256(downloaded)==expected_sha
    try:
        download_asset(FakeSession(binary[:-1]+b'x'),{'id':1},info,td/'corrupted.xlsx')
    except ValueError as exc:
        assert 'КОНТРОЛЬНАЯ СУММА' in str(exc)
    else:raise AssertionError('SHA canary failed to detect corruption')
    assert not (td/'corrupted.xlsx').exists()
print('OFFLINE PREFLIGHT: XLSX + GitHub binary streaming + SHA256 corruption canary — PASS')

from_github.xlsx:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

SHA256 PASS: from_github.xlsx 7350641cdde3d78305b3a90d5fe95b563de58365ac9528df4d07e60dba07b906


corrupted.xlsx:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

OFFLINE PREFLIGHT: XLSX + GitHub binary streaming + SHA256 corruption canary — PASS


## 1. Проверка GitHub Release и загрузка настоящих файлов

Проверяется совпадение **tag → commit → manifest → asset**. Права GitHub проверяются до загрузки, размеры сверяются с manifest, тело скачивается потоково, полный SHA-256 считается по байтам. Файл неверного размера либо с неверной суммой отвергается. Повторный запуск использует локальный кэш **только после повторной проверки SHA**.

In [4]:
# This first phase never changes GitHub or the published run.
TOKEN=get_token()
SESSION=session_for(TOKEN)
SOURCE_MANIFEST,ASSETS,COMMIT,RELEASE_URL=github_manifest_and_release(SESSION)
PRED_FILE=download_asset(SESSION,ASSETS[PRED_NAME],SOURCE_MANIFEST['outputs'][PRED_NAME],INPUT_DIR/PRED_NAME)
RESULTS_FILE=download_asset(SESSION,ASSETS[RESULTS_NAME],SOURCE_MANIFEST['outputs'][RESULTS_NAME],INPUT_DIR/RESULTS_NAME)
SESSION.close();del TOKEN

Зафиксированная публикация: https://github.com/AntonDozhdikov/Demograf_sovetnik/releases/tag/stage2-ml1-v4.0.0-20261026T130048Z-58bd8971
Git commit: 462a92a0f3ae4d52c2a023a3f76af611adda6062


ML1_V4_PREDICTIONS.xlsx:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

SHA256 PASS: ML1_V4_PREDICTIONS.xlsx fe877e5fa0d65ffda2acd3f6d9cb40e3af7c1b1242d20d43b92f46710a713350


ML1_V4_RESULTS.xlsx:   0%|          | 0.00/981k [00:00<?, ?B/s]

SHA256 PASS: ML1_V4_RESULTS.xlsx 26e13ba5c40d0efb0565fd8be3f92d2d6bbcf770f62fb77b043365a1d78b3e78


## 2. Контракт данных и полный пересчёт регрессионных метрик

Основная единица сравнения: `(horizon, protocol, feature_set, model, seed, row_id)`. Дубликаты, пропущенные значения, изменённая истина между моделями и нарушения информационных срезов приводят к остановке исследования, а не к молчаливому удалению строк.

Показатели пересчитываются непосредственно из `target_rate` и `prediction`, затем сопоставляются **для каждой конфигурации** с опубликованным `Модельные_метрики` по MAE, RMSE, bias, 90-му перцентилю ошибки и взвешенному MAE. Целевой коэффициент — человек на 1 000 жителей.

In [5]:
# 02 — Read the REAL V4 schema. Streaming XLSX input to bound transient memory.
NUMERIC=['origin_year','target_year','information_cutoff','max_feature_year','horizon',
         'target_rate','prediction','seed','max_label_year_fit','population_exposure']
BASE_COLS=['row_id','territory_id','region_id','mun_type','origin_year','target_year',
           'information_cutoff','max_feature_year','horizon','target_rate','prediction',
           'model','feature_set','protocol','seed','max_label_year_fit','population_exposure']
KEY=['horizon','protocol','feature_set','model','seed']
CTX=['horizon','protocol']
QA_ROWS=[]
def qa(level,code,detail,n=0):
    QA_ROWS.append({'уровень':level,'код':code,'описание':str(detail)[:300],'число':int(n)})

def xlsx_rows(file,sheet,need=None,expected=None,bar=True):
    book=load_workbook(file,read_only=True,data_only=True)
    try:
        if sheet not in book.sheetnames:raise KeyError(f'Отсутствует лист {sheet!r} — есть {book.sheetnames}')
        ws=book[sheet]
        it=ws.iter_rows(values_only=True)
        header=[str(x).strip() if x is not None else '' for x in next(it)]
        if len(header)!=len(set(header)) or '' in header:raise ValueError('Неверная/дублирующаяся шапка '+sheet)
        missing=set(need or [])-set(header)
        if missing:raise ValueError(f'{sheet}: обязательные столбцы отсутствуют: {missing}')
        total=0
        with tqdm(total=expected,desc=sheet,unit='стр',disable=not bar) as progress:
            for raw in it:
                if not any(x is not None for x in raw):continue
                total+=1
                progress.update()
                yield dict(zip(header,raw))
        if expected is not None and total!=expected:
            raise ValueError(f'{sheet}: {total} строк, по манифесту {expected}')
    finally: book.close()

def read_small(file,sheet):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',UserWarning)
        return pd.read_excel(file,sheet_name=sheet,engine='openpyxl')

def load_prediction_contexts():
    by_h={};checks=[]
    for h in (1,3,5):
        sheet=f'Прогноз_h{h}'
        manifest_sheet=SOURCE_MANIFEST['outputs'][PRED_NAME]['sheets'][sheet]
        expected=manifest_sheet['rows']
        buf=[]
        for row in xlsx_rows(PRED_FILE,sheet,need=BASE_COLS,expected=expected):
            buf.append([row.get(k) for k in BASE_COLS])
        df=pd.DataFrame.from_records(buf,columns=BASE_COLS);del buf
        for c in NUMERIC:df[c]=pd.to_numeric(df[c],errors='coerce')
        # Preserve exact territorial strings and row IDs. Never cast them to int.
        for c in ['row_id','territory_id','region_id','mun_type','model','feature_set','protocol']:
            df[c]=df[c].astype('string')
        checks.append({'лист':sheet,'строк':len(df),'полей':df.shape[1],
                       'SHA256_книги':SOURCE_MANIFEST['outputs'][PRED_NAME]['sha256']})
        if not (df.horizon==h).all():qa('CRITICAL','HORIZON',f'{sheet}: другое horizon',int((df.horizon!=h).sum()))
        by_h[h]=df
    return pd.concat(list(by_h.values()),ignore_index=True),pd.DataFrame(checks)

PRED,SHEET_AUDIT=load_prediction_contexts()
print('Основных прогнозов загружено:',len(PRED))
assert len(PRED)==SOURCE_MANIFEST['n_prediction_rows']
PUB_METRICS=read_small(RESULTS_FILE,'Модельные_метрики')
PUB_HYPOTHESES=read_small(RESULTS_FILE,'Гипотезы_H1_H3')
PUB_STATTEST=read_small(RESULTS_FILE,'Стат_тесты')
PUB_FLOWS=read_small(RESULTS_FILE,'Потоки')
PUB_CLASS=read_small(RESULTS_FILE,'Классификация')
PUB_INTERVAL=read_small(RESULTS_FILE,'Прогнозные_интервалы')
# All reported V4 groups must be accounted for, not just DEV-selected rows.
GROUPED=PRED.groupby(KEY,dropna=False,observed=True,sort=False)
GROUP_SIZES=GROUPED.agg(n=('row_id','size'),unique_keys=('row_id','nunique'),
                            min_truth=('target_rate','min'),max_truth=('target_rate','max')).reset_index()
if GROUP_SIZES['n'].sum()!=len(PRED):qa('CRITICAL','GROUP_SUM','Группировка потеряла строки')
duplicates=int((GROUP_SIZES.n-GROUP_SIZES.unique_keys).sum())
if duplicates:qa('CRITICAL','KEY_DUPLICATES','Дубли row_id внутри model/seed/feature/protocol',duplicates)
if PRED[KEY+['row_id']].isna().any().any():
    qa('CRITICAL','MISSING_IDS','Пропущенные идентификаторы',int(PRED[KEY+['row_id']].isna().any(axis=1).sum()))
for c in ['target_rate','prediction','horizon','origin_year','target_year','seed']:
    bad=int((~np.isfinite(pd.to_numeric(PRED[c],errors='coerce').to_numpy(float))).sum())
    if bad:qa('CRITICAL','NONFINITE_'+c,c,bad)
for c in ['max_feature_year','information_cutoff','max_label_year_fit']:
    missing=int(PRED[c].isna().sum())
    if missing:qa('CRITICAL','MISSING_'+c,c,missing)

LEAK_FLAGS={
 'future_feature':(PRED.max_feature_year>PRED.information_cutoff),
 'future_training_label':(PRED.max_label_year_fit>PRED.information_cutoff),
 'incorrect_horizon':((PRED.target_year-PRED.origin_year)!=PRED.horizon),
 'feature_after_origin':(PRED.max_feature_year>PRED.origin_year),
 'missing_population':PRED.population_exposure.isna(),
 'nonpositive_population':(PRED.population_exposure<=0),
}
for code,mask in LEAK_FLAGS.items():
    n=int(mask.fillna(False).sum())
    if n:qa('CRITICAL' if code in ('future_feature','future_training_label','incorrect_horizon','feature_after_origin') else 'WARNING',code,code,n)

# Prediction for one row_id must have the same truth, territory, target time across models.
cons=PRED.groupby(CTX+['row_id'],observed=True).agg(
    truth_min=('target_rate','min'),truth_max=('target_rate','max'),
    target_year_min=('target_year','min'),target_year_max=('target_year','max'),
    territories=('territory_id','nunique'),regions=('region_id','nunique')).reset_index()
conflict=((cons.truth_max-cons.truth_min).abs()>1e-7)|(cons.target_year_min!=cons.target_year_max)|(cons.territories!=1)|(cons.regions!=1)
if conflict.any():qa('CRITICAL','TRUTH_CONFLICT','Разные истины или территория для одного TEST row_id',int(conflict.sum()))
CONSISTENCY_BY_CONTEXT=cons.assign(conflict=conflict).groupby(CTX,as_index=False).agg(
    n_unique_rows=('row_id','size'),contradictions=('conflict','sum'))
# Key coverage can vary legitimately; explicitly record all deviations.
coverage=GROUP_SIZES.merge(CONSISTENCY_BY_CONTEXT,on=CTX,how='left')
coverage['missing_context_keys']=coverage.n_unique_rows-coverage.unique_keys
if (coverage.missing_context_keys>0).any():
    qa('WARNING','PARTIAL_KEY_COVERAGE','Некоторые модели не охватывают полный набор TEST ключей',int((coverage.missing_context_keys>0).sum()))
COVERAGE=coverage

# Hard-stop for invalid claims. Real inference never silently drops duplicate, missing or leaked observations.
critical=[x for x in QA_ROWS if x['уровень']=='CRITICAL']
if critical:
    raise AssertionError('НАРУШЕН НАУЧНЫЙ КОНТРАКТ ПРОГНОЗОВ: '+repr(critical[:8]))
print('Контракт row_id / времён / источников: PASS')
print('Групп моделей, включая seeds:',len(GROUP_SIZES))

Прогноз_h1:   0%|          | 0/290778 [00:00<?, ?стр/s]

Прогноз_h3:   0%|          | 0/290410 [00:00<?, ?стр/s]

Прогноз_h5:   0%|          | 0/289472 [00:00<?, ?стр/s]

Основных прогнозов загружено: 870660
Контракт row_id / времён / источников: PASS
Групп моделей, включая seeds: 486


## 3. Выбор по DEV и контролируемые сравнения

**Выполняется неизменный выбор на DEV**, а TEST используется только для оценки:

\[
MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|,
\qquad
\Delta MAE=MAE_{base}-MAE_{candidate}.
\]

Положительное \(\Delta MAE\) — улучшение. Парные сравнения разрешены только при **полностью одинаковом множестве `row_id`**. Доверительные интервалы строятся региональным блочным bootstrap по регионам, а для дополнительных множественных *post hoc* сравнений рассчитывается FDR Бенджамини—Хохберга. Это не подтверждение причинности; случайная величина здесь — прогнозная ошибка.

In [6]:
# 03 — Complete replay of V4 regression metrics, DEV-locked selections and paired tests
PRED['signed_error']=PRED.prediction-PRED.target_rate
PRED['absolute_error']=PRED.signed_error.abs()
PRED['squared_error']=PRED.signed_error**2
PRED['weighted_absolute_error']=np.where(PRED.population_exposure.gt(0),
    PRED.absolute_error*PRED.population_exposure,0.)
PRED['valid_weight']=np.where(PRED.population_exposure.gt(0),PRED.population_exposure,0.)

raw=PRED.groupby(KEY,dropna=False,observed=True,sort=False).agg(
    n=('row_id','size'),MAE=('absolute_error','mean'),MSE=('squared_error','mean'),
    bias=('signed_error','mean'),p90_abs_error=('absolute_error',lambda s:float(s.quantile(.90))),
    weighted_err=('weighted_absolute_error','sum'),valid_weight=('valid_weight','sum')).reset_index()
raw['RMSE']=np.sqrt(raw.MSE)
raw['weighted_MAE']=raw.weighted_err/raw.valid_weight.replace(0,np.nan)
RECALC=raw.drop(columns=['MSE','weighted_err','valid_weight']).copy()
# Both V4 test tables and replay use this same unrounded metric definition.
for k in ['horizon','seed']:PUB_METRICS[k]=pd.to_numeric(PUB_METRICS[k],errors='coerce')
metric_join=PUB_METRICS.merge(RECALC,on=KEY,how='outer',suffixes=('_V4','_replay'),indicator=True,validate='one_to_one')
if not metric_join['_merge'].eq('both').all():
    qa('CRITICAL','METRIC_KEY_PARITY','Прогнозы и опубликованные метрики имеют несовпадающие конфигурации',int((metric_join['_merge']!='both').sum()))
for metric in ('n','MAE','RMSE','bias','p90_abs_error','weighted_MAE'):
    a=pd.to_numeric(metric_join[metric+'_V4'],errors='coerce')
    b=pd.to_numeric(metric_join[metric+'_replay'],errors='coerce')
    delta=(a-b).abs();metric_join['delta_'+metric]=delta
    fail=delta.gt(1e-7) | (a.isna() ^ b.isna())
    if fail.any():qa('CRITICAL','METRIC_MISMATCH_'+metric,f'Пересчёт {metric} не совпадает с V4',int(fail.sum()))
if len(RECALC)!=len(PUB_METRICS):qa('CRITICAL','METRICS_COUNT','Количество строк метрик отличается от исходного')
METRIC_PARITY=metric_join[[*KEY,'_merge']+[x for x in metric_join.columns if x.startswith('delta_')]].copy()
if any(x['уровень']=='CRITICAL' for x in QA_ROWS):
    raise AssertionError('Сверка всех модельных метрик не пройдена: '+repr([x for x in QA_ROWS if x['уровень']=='CRITICAL'][-5:]))
print(f'МЕТРИКИ: {len(RECALC)}/{len(PUB_METRICS)} строк совпали с V4')

# Only original DEV measurements determine the chosen model (never choose on TEST).
choices=PUB_METRICS[(PUB_METRICS.seed==11)&pd.to_numeric(PUB_METRICS.MAE_DEV,errors='coerce').notna()].copy()
choices=choices.sort_values(['MAE_DEV','model','feature_set'])
DEV_WIN=choices.groupby(CTX,observed=True,as_index=False).head(1).reset_index(drop=True)
DEV_FAMILY_WIN=choices.groupby(CTX+['feature_set'],observed=True,as_index=False).head(1).reset_index(drop=True)
if len(DEV_WIN)!=12:qa('CRITICAL','DEV_SELECTION','Ожидалось 12 контекстов DEV, получено '+str(len(DEV_WIN)))

# Sample parity is strict; no pairwise calculation on unpaired observations.
REGION_COLS=['row_id','target_rate','prediction','region_id','territory_id','target_year','mun_type']
def frame_model(h,p,model,family,seed=11):
    q=PRED.loc[(PRED.horizon==h)&PRED.protocol.eq(p)&PRED.model.eq(model)&
                 PRED.feature_set.eq(family)&PRED.seed.eq(seed),REGION_COLS]
    if len(q)==0:return None
    if q.row_id.duplicated().any():raise ValueError('Duplicate row ID for '+str((h,p,model,family,seed)))
    return q

def pair_versus(h,protocol,cand_model,cand_family,base_model,base_family,cand_seed=11,base_seed=11):
    c=frame_model(h,protocol,cand_model,cand_family,cand_seed)
    b=frame_model(h,protocol,base_model,base_family,base_seed)
    if c is None or b is None:return None, 'MISSING_MODEL'
    if len(c)!=len(b) or set(c.row_id)!=set(b.row_id):return None,'NON_IDENTICAL_TEST_KEYS'
    m=c.merge(b[['row_id','target_rate','prediction']],on='row_id',how='inner',validate='one_to_one',suffixes=('_cand','_base'))
    if len(m)!=len(c) or not np.allclose(m.target_rate_cand,m.target_rate_base,rtol=0,atol=1e-7):
        return None,'MISMATCHED_TARGETS'
    m['difference']=abs(m.prediction_base-m.target_rate_cand)-abs(m.prediction_cand-m.target_rate_cand)
    return m,'OK'

def region_ci(m,iterations=BOOTSTRAPS,seed=BOOT_SEED):
    # Cluster re-sampling respects municipality/region correlation: one region = one block.
    if m is None or m.empty:return {'status':'NO_PAIRS'}
    g=m.groupby('region_id',dropna=False).difference.agg(['sum','count'])
    if len(g)<5:return {'status':'NOT_FEASIBLE_LT5_REGIONS','regions':len(g)}
    s=g['sum'].to_numpy(float);n=g['count'].to_numpy(float)
    obs=float(s.sum()/n.sum())
    rng=np.random.default_rng(seed)
    ids=rng.integers(0,len(g),size=(iterations,len(g)))
    res=s[ids].sum(axis=1)/n[ids].sum(axis=1)
    lo,hi=np.quantile(res,[.025,.975])
    centered=res-obs
    p=(1+np.sum(abs(centered)>=abs(obs)))/(iterations+1)
    return {'status':'OK','delta_MAE':obs,'CI_low':float(lo),'CI_high':float(hi),
            'p_boot_centered':float(min(1,p)),'regions':len(g),'n':len(m)}

def bh_adjust(series):
    p=np.asarray(series,float);finite=np.isfinite(p);z=np.full(p.shape,np.nan)
    if not finite.any():return z
    inds=np.where(finite)[0];order=inds[np.argsort(p[finite])];rank=np.arange(1,len(order)+1)
    adjust=np.minimum.accumulate((p[order]*len(order)/rank)[::-1])[::-1]
    z[order]=np.minimum(adjust,1)
    return z

# Benchmark vs last-known: DEV-selected best per context, plus best DEV per factor group.
BENCHMARK=[]
for kind,choice in [('GLOBAL_DEV',DEV_WIN),('FAMILY_DEV',DEV_FAMILY_WIN)]:
    for _,r in choice.iterrows():
        h,proto=int(r.horizon),str(r.protocol)
        m,status=pair_versus(h,proto,r.model,r.feature_set,'last_known','A_history',11,0)
        extra=region_ci(m,seed=BOOT_SEED+int(h)) if status=='OK' else {'status':status}
        if m is not None:
            extra['MAE_baseline']=float(abs(m.prediction_base-m.target_rate_cand).mean())
            extra['MAE_candidate_replay']=float(abs(m.prediction_cand-m.target_rate_cand).mean())
            extra['improvement_percent']=100*extra['delta_MAE']/extra['MAE_baseline'] if extra['MAE_baseline']>0 else np.nan
        BENCHMARK.append({'selection':kind,'horizon':h,'protocol':proto,'model':r.model,
           'feature_set':r.feature_set,'MAE_DEV':r.MAE_DEV,
           'baseline':'last_known','MAE_model':float(r.MAE),'n_model':int(r.n),**extra})
BASELINE_COMP=pd.DataFrame(BENCHMARK)
BASELINE_COMP['p_fdr_within_selection']=np.nan
for sel in BASELINE_COMP.selection.unique():
    ix=BASELINE_COMP.index[BASELINE_COMP.selection==sel]
    BASELINE_COMP.loc[ix,'p_fdr_within_selection']=bh_adjust(BASELINE_COMP.loc[ix,'p_boot_centered'])

# Publication-critical ablations: SAME estimator/seed/test keys, changing factor group only.
TRANSITIONS=[('H1','A_history','B_economy'),('H2','B_economy','C_housing_services'),
             ('H3','C_housing_services','D_spatial'),('E_vs_A','A_history','E_all'),
             ('E_vs_C','C_housing_services','E_all')]
ABL=[]
for h,proto in sorted(PRED[CTX].drop_duplicates().itertuples(index=False,name=None)):
    opts=set(choices.loc[(choices.horizon==h)&(choices.protocol==proto),'model'])
    for model in sorted(opts):
        fs_avail=set(choices.loc[(choices.horizon==h)&(choices.protocol==proto)&
                                  (choices.model==model),'feature_set'])
        for hypoth,basefs,altfs in TRANSITIONS:
            if not {basefs,altfs}.issubset(fs_avail):continue
            m,status=pair_versus(h,proto,model,altfs,model,basefs)
            extra=region_ci(m,seed=BOOT_SEED+h+sum(map(ord,model))) if status=='OK' else {'status':status}
            if m is not None:
                extra['MAE_base']=float(abs(m.prediction_base-m.target_rate_cand).mean())
                extra['MAE_added']=float(abs(m.prediction_cand-m.target_rate_cand).mean())
                extra['improvement_percent']=100*extra['delta_MAE']/extra['MAE_base'] if extra['MAE_base']>0 else np.nan
            ABL.append({'comparison':hypoth,'horizon':int(h),'protocol':str(proto),'algorithm':str(model),
                        'base_features':basefs,'add_features':altfs,'same_algorithm':True,**extra})
ABLATIONS=pd.DataFrame(ABL)
if ABLATIONS.empty:qa('WARNING','NO_ABLATIONS','Нет допустимых абляций при одном алгоритме')
else:
    ABLATIONS['p_fdr_exploratory']=np.nan
    # Correct across ALL ex-post pair comparisons (not 12 pre-registered H1/H2 tests).
    ABLATIONS['p_fdr_exploratory']=bh_adjust(ABLATIONS['p_boot_centered'])
    ABLATIONS['positive_CI']=ABLATIONS.CI_low.gt(0)
# 50 seeds ONLY in the intended ROLLING_TIME contexts, not in all 12 contexts.
SEEDS=RECALC[RECALC.seed.ge(1000)].copy()
STABILITY=SEEDS.groupby(CTX+['feature_set','model'],as_index=False,observed=True).agg(
    n_seeds=('seed','nunique'),MAE_mean=('MAE','mean'),MAE_std=('MAE','std'),
    MAE_min=('MAE','min'),MAE_max=('MAE','max'),MAE_p025=('MAE',lambda s:s.quantile(.025)),
    MAE_p975=('MAE',lambda s:s.quantile(.975)))
if len(SEEDS)!=150:qa('WARNING','SEED_COUNT',f'Expected 150 seed rows, received {len(SEEDS)}')

# Breakdown tables for publication, no model selection on TEST.
YEARLY=PRED.groupby(KEY+['target_year'],observed=True,as_index=False).agg(
    n=('row_id','size'),MAE=('absolute_error','mean'),bias=('signed_error','mean'))
REGIONAL=PRED.groupby(KEY+['region_id'],observed=True,as_index=False).agg(
    n=('row_id','size'),MAE=('absolute_error','mean'),bias=('signed_error','mean'))
TYPES=PRED.groupby(KEY+['mun_type'],observed=True,as_index=False).agg(
    n=('row_id','size'),MAE=('absolute_error','mean'),bias=('signed_error','mean'))
# All by-segment tables are descriptive; no post-hoc region ranking causal claims.
print('DEV winners:',len(DEV_WIN),'| paired baseline:',len(BASELINE_COMP),'| same-model ablations:',len(ABLATIONS))
print('50-seed contexts:',len(STABILITY),'| regional metrics:',len(REGIONAL))

МЕТРИКИ: 486/486 строк совпали с V4
DEV winners: 12 | paired baseline: 60 | same-model ablations: 288
50-seed contexts: 3 | regional metrics: 29238


## 4. Остальные типы прогнозов

Реально существующие листы `Абс_потоки`, `Классиф_прогнозы` и `Интервальные_прогнозы` проверяются самостоятельно. Для абсолютных потоков пересчитываются MAE/RMSE в людях, для классификации — Brier, ROC-AUC, PR-AUC и калибровочные группы; для интервалов — фактическое покрытие, средняя ширина и биномиальный интервал Уилсона (ограниченно применимый при пространственной зависимости). **Рекурсивное правило не маркируется как калиброванная вероятностная модель.**

In [8]:
# 04 — Проверка потоков, классификации и интервалов V4
_AUX_PREFIX = ('FLOW_', 'FLOWS_', 'CLASS_', 'INTERVAL_')
QA_ROWS[:] = [r for r in QA_ROWS if not r['код'].startswith(_AUX_PREFIX)]

def aux(sheet):
    n = SOURCE_MANIFEST['outputs'][PRED_NAME]['sheets'][sheet]['rows']
    d = pd.DataFrame(xlsx_rows(PRED_FILE, sheet, expected=n))
    for c in ('horizon','truth','prediction','actual','probability','low','high','covered'):
        if c in d: d[c] = pd.to_numeric(d[c], errors='coerce')
    print(f'{sheet}: {len(d):,} строк'); return d

def audit_keys(d, keys, required, prefix):
    absent = set(keys + required) - set(d)
    if absent: raise ValueError(f'{prefix}: отсутствуют столбцы {sorted(absent)}')
    bad = int(d.duplicated(keys).sum())
    if bad: qa('CRITICAL', prefix+'_KEYS', 'Повторяющиеся ключи', bad)

def parity(calc, published, keys, prefix):
    out = calc.merge(published, on=keys, how='outer', suffixes=('_replay','_V4'), indicator=True, validate='one_to_one')
    bad = int(out['_merge'].ne('both').sum())
    if bad: qa('CRITICAL', prefix+'_META', 'Несовпадение контекстов', bad)
    return out

FLOW_PRED, CLS_PRED, INT_PRED = (aux(s) for s in ('Абс_потоки','Классиф_прогнозы','Интервальные_прогнозы'))
FG, CG, IG = ['horizon','protocol','outcome','model'], ['horizon','protocol','target','model'], ['horizon','protocol','model']

# I. Абсолютные потоки
audit_keys(FLOW_PRED, FG+['row_id'], ['truth','prediction'], 'FLOW')
v = FLOW_PRED[['truth','prediction']].to_numpy(float); bad = int((~np.isfinite(v)).any(axis=1).sum())
if bad: qa('CRITICAL','FLOW_NONFINITE','NaN/Inf в потоках',bad); raise ValueError('Некорректные потоки')

FLOW_PRED['error'] = FLOW_PRED.prediction - FLOW_PRED.truth
FLOW_PRED['abs_error'], FLOW_PRED['square_error'] = FLOW_PRED.error.abs(), FLOW_PRED.error**2
FLOW_RECALC = FLOW_PRED.groupby(FG,observed=True,as_index=False).agg(n=('row_id','size'),MAE_persons=('abs_error','mean'),MSE=('square_error','mean'),bias=('error','mean'))
FLOW_RECALC['RMSE_persons'] = np.sqrt(FLOW_RECALC.pop('MSE'))
FLOWS_PARITY = parity(FLOW_RECALC, PUB_FLOWS.rename(columns={'target':'outcome'}), FG, 'FLOWS')

for m in ('MAE_persons','RMSE_persons'):
    a,b = (pd.to_numeric(FLOWS_PARITY[m+s],errors='coerce').to_numpy(float) for s in ('_replay','_V4'))
    delta = np.abs(a-b); limit = 1e-6 + 1e-12*np.maximum(np.abs(a),np.abs(b))
    failed = ~np.isfinite(a) | ~np.isfinite(b) | (delta > limit)
    FLOWS_PARITY['delta_'+m], FLOWS_PARITY['tolerance_'+m], FLOWS_PARITY['status_'+m] = delta, limit, np.where(failed,'FAIL','PASS')
    if failed.any(): qa('CRITICAL','FLOWS_METRIC_'+m,'Разница превышает масштабный допуск',int(failed.sum()))
    print(f'{m}: {len(failed)-failed.sum()}/{len(failed)} PASS; относительная ошибка ≤ {np.nanmax(delta/np.maximum(np.maximum(np.abs(a),np.abs(b)),1)):.2e}')

unstable = FLOW_RECALC.model.astype(str).str.contains('recursive_ridge',case=False,regex=False) & FLOW_RECALC.MAE_persons.gt(1e6)
if unstable.any(): qa('WARNING','FLOWS_RECURSIVE_RIDGE_INSTABILITY','Неустойчивые потоки Ridge',int(unstable.sum()))

# II. Классификация и калибровка
audit_keys(CLS_PRED,CG+['row_id'],['truth','probability'],'CLASS')
p,y = CLS_PRED.probability.to_numpy(float), CLS_PRED.truth.to_numpy(float)
bad = int((~np.isfinite(p) | (p<0) | (p>1) | ~np.isin(y,[0,1])).sum())
if bad: qa('CRITICAL','CLASS_PROBABILITY','Некорректные вероятности/классы',bad); raise ValueError('Некорректные вероятности')

cr,calib = [],[]
for key,g in CLS_PRED.groupby(CG,observed=True,sort=False):
    p,y = g.probability.to_numpy(float),g.truth.to_numpy(float); good = len(np.unique(y))==2; base = dict(zip(CG,key))
    cr.append({**base,'n':len(g),'prevalence':float(y.mean()),'brier':float(brier_score_loss(y,p)),
               'ROC_AUC':float(roc_auc_score(y,p)) if good else np.nan,'PR_AUC':float(average_precision_score(y,p)) if y.sum()>0 else np.nan,
               'status':'OK' if good else 'ONLY_ONE_CLASS','probability_min':float(p.min()),'probability_max':float(p.max())})
    bins = np.clip((10*p).astype(int),0,9)
    for j in range(10):
        mask = bins==j
        if mask.any(): calib.append({**base,'bin':f'{j/10:.1f}–{(j+1)/10:.1f}','n':int(mask.sum()),
                                     'observed_frequency':float(y[mask].mean()),'mean_probability':float(p[mask].mean())})

CLASS_RECALC, CALIBRATION = pd.DataFrame(cr),pd.DataFrame(calib)
CLASS_PARITY = parity(CLASS_RECALC,PUB_CLASS,CG,'CLASS')
for m in ('brier','ROC_AUC','PR_AUC'):
    if m+'_V4' not in CLASS_PARITY: continue
    a,b = (pd.to_numeric(CLASS_PARITY[m+s],errors='coerce') for s in ('_replay','_V4'))
    d = (a-b).abs(); CLASS_PARITY['delta_'+m] = d
    if d.gt(1e-6).any(): qa('WARNING','CLASS_METRIC_'+m,'Несовпадение с V4',int(d.gt(1e-6).sum()))

# III. Прогнозные интервалы
audit_keys(INT_PRED,IG+['row_id'],['actual','low','high'],'INTERVAL')
a,lo,hi = (INT_PRED[c].to_numpy(float) for c in ('actual','low','high'))
bad = int((~np.isfinite(a) | ~np.isfinite(lo) | ~np.isfinite(hi) | (lo>hi)).sum())
if bad: qa('CRITICAL','INTERVAL_BOUNDS','NaN/Inf или неверные границы',bad); raise ValueError('Некорректные интервалы')

INT_PRED['recomputed_covered'] = (INT_PRED.actual>=INT_PRED.low)&(INT_PRED.actual<=INT_PRED.high)
INT_PRED['width'] = INT_PRED.high-INT_PRED.low
if 'covered' in INT_PRED:
    bad = int(INT_PRED.covered.ne(INT_PRED.recomputed_covered.astype(float)).sum())
    if bad: qa('CRITICAL','INTERVAL_COVERAGE_ROWS','Неверный флаг покрытия',bad)

INT_RECALC = INT_PRED.groupby(IG,observed=True,as_index=False).agg(n=('row_id','size'),coverage=('recomputed_covered','mean'),mean_width=('width','mean'))
z = norm.ppf(.975); n,p = INT_RECALC.n.to_numpy(float),INT_RECALC.coverage.to_numpy(float); denom = 1+z*z/n
ctr = (p+z*z/(2*n))/denom; half = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/denom
INT_RECALC['Wilson_low'],INT_RECALC['Wilson_high'] = ctr-half,ctr+half
INT_RECALC['nominal'],INT_RECALC['coverage_gap'] = .9,p-.9
INTERVAL_PARITY = parity(INT_RECALC,PUB_INTERVAL,IG,'INTERVAL')

for a,b in [('coverage','empirical_coverage'),('mean_width','mean_width_pp1000')]:
    d = (INTERVAL_PARITY[a]-INTERVAL_PARITY[b]).abs()
    if d.gt(1e-7).any(): qa('CRITICAL','INTERVAL_METRIC_'+a,'Несовпадение с V4',int(d.gt(1e-7).sum()))

truth = PRED.loc[PRED.seed.eq(11),CTX+['row_id','target_rate']].drop_duplicates()
divergent = truth.groupby(CTX+['row_id']).target_rate.nunique(dropna=False).gt(1)
if divergent.any(): qa('CRITICAL','INTERVAL_TRUTH_NONUNIQUE','Разные фактические значения row_id',int(divergent.sum()))
joined = INT_PRED.merge(truth.drop_duplicates(CTX+['row_id']),on=CTX+['row_id'],how='left',validate='many_to_one')
missing = int(joined.target_rate.isna().sum()); mismatch = int((joined.actual-joined.target_rate).abs().gt(1e-7).sum())
if missing: qa('CRITICAL','INTERVAL_MISSING_TRUTH','Отсутствующие row_id',missing)
if mismatch: qa('CRITICAL','INTERVAL_TRUTH_MISMATCH','Несовпадение фактических значений',mismatch)

critical = [r for r in QA_ROWS if r['уровень']=='CRITICAL']
if critical: raise AssertionError('Расхождение V4: '+repr(critical[-8:]))
print(f'АУДИТ PASS | потоки {len(FLOW_RECALC)} | классификация {len(CLASS_RECALC)} | интервалы {len(INT_RECALC)} | предупреждения {sum(r["уровень"]=="WARNING" for r in QA_ROWS)}')


Абс_потоки:   0%|          | 0/115464 [00:00<?, ?стр/s]

Абс_потоки: 115,464 строк


Классиф_прогнозы:   0%|          | 0/76921 [00:00<?, ?стр/s]

Классиф_прогнозы: 76,921 строк


Интервальные_прогнозы:   0%|          | 0/6426 [00:00<?, ?стр/s]

Интервальные_прогнозы: 6,426 строк
MAE_persons: 72/72 PASS; относительная ошибка ≤ 4.77e-16
RMSE_persons: 72/72 PASS; относительная ошибка ≤ 6.05e-16
АУДИТ PASS | потоки 72 | классификация 48 | интервалы 4 | предупреждения 1


## 5. Графики и публикационные таблицы

Формируются 10 отдельных графиков и два Excel: `ML1_V4_PREDICTIONS_PUBLICATION.xlsx` (метрики, DEV-выбор, сравнения, регионы, годы, типологии, вероятность, интервалы) и `ML1_V4_PREDICTIONS_QA.xlsx` (полный след проверки, расхождения, непокрытые TEST-ключи, происхождение и ограничения). Каждый лист после записи проверяется повторным чтением, а архив рисунков — по CRC.

In [9]:
# 05 — Publication-ready figures and two consolidated Excel workbooks
from textwrap import shorten
import xlsxwriter

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':12,
                     'savefig.dpi':FIGURE_DPI,'axes.grid':True,'grid.alpha':0.25})
FIGURES=[]
def figure(name,plotter):
    fig,ax=plt.subplots(figsize=(10,5.6),layout='constrained')
    try:
        plotter(ax)
        fig.savefig(PLOTS_DIR/name,dpi=FIGURE_DPI,bbox_inches='tight',facecolor='white')
        FIGURES.append(PLOTS_DIR/name)
    finally:plt.close(fig)

# 01 baseline comparison: selected on DEV, displayed on held-out TEST.
def draw_baseline(ax):
    d=BASELINE_COMP[(BASELINE_COMP.selection=='GLOBAL_DEV')&(BASELINE_COMP.status=='OK')].copy()
    d=d.sort_values(['horizon','protocol'])
    names=[f'{int(r.horizon)} г. · {r.protocol}' for r in d.itertuples()]
    y=np.arange(len(d));delta=d.delta_MAE.to_numpy(float)
    ax.barh(y,delta,color=np.where(delta>=0,'#287c80','#b46464'))
    ax.set_yticks(y,names);ax.axvline(0,color='#555',lw=1)
    ax.invert_yaxis();ax.set_xlabel('Снижение MAE относительно инерционного прогноза, чел./1000')
    ax.set_title('Модели, выбранные на DEV: выигрыш на закрытом TEST')
figure('01_DEV_выбор_против_инерции.png',draw_baseline)

# 02 all six estimators, no claim that TEST minimum is a deployable selection.
def draw_models(ax):
    d=RECALC[(RECALC.seed==11)&(RECALC.protocol=='ROLLING_TIME')]
    d=d[~d.model.str.contains('last_known|mean3|median',case=False,regex=True,na=False)]
    d=d.groupby(['horizon','model'],as_index=False).MAE.mean()
    for model,part in d.groupby('model'):
        ax.plot(part.horizon,part.MAE,marker='o',label=model)
    ax.set_xticks([1,3,5]);ax.set_xlabel('Горизонт прогноза, лет');ax.set_ylabel('Средний MAE по факторным конфигурациям')
    ax.set_title('Алгоритмы: описательное сравнение в ROLLING_TIME')
    ax.legend(fontsize=8,ncol=2)
figure('02_алгоритмы_горизонты.png',draw_models)

# 03 forest of SAME-ALGORITHM feature ablations.
def draw_ablation(ax):
    if ABLATIONS.empty:
        ax.text(.5,.5,'Нет допустимых пар по одинаковому алгоритму',ha='center',va='center',transform=ax.transAxes)
        return
    d=ABLATIONS[ABLATIONS.status=='OK'].copy().sort_values('delta_MAE',ascending=False)
    d=d.head(24)
    if not len(d):return
    yy=np.arange(len(d))
    ax.errorbar(d.delta_MAE,yy,xerr=[d.delta_MAE-d.CI_low,d.CI_high-d.delta_MAE],fmt='o',color='#276b72',capsize=2)
    ax.set_yticks(yy,[f'{r.comparison} · h{int(r.horizon)} · {r.protocol} · {r.algorithm}' for r in d.itertuples()],fontsize=7)
    ax.invert_yaxis();ax.axvline(0,color='gray',lw=1)
    ax.set_xlabel('Улучшение MAE при добавлении факторов, чел./1000')
    ax.set_title('Контролируемые абляции: один алгоритм, одинаковые TEST row_id')
figure('03_контролируемые_абляции.png',draw_ablation)

# 04 50-seed dispersion (only where run was actually performed).
def draw_seeds(ax):
    d=SEEDS[SEEDS.protocol=='ROLLING_TIME']
    for (h,model),part in d.groupby(['horizon','model']):
        yy=part.MAE.to_numpy()
        ax.scatter(np.repeat(int(h),len(yy))+np.random.default_rng(h).normal(0,.018,len(yy)),yy,
                   label=f'h{h} {model}',alpha=.45,s=10)
    ax.set_xticks([1,3,5]);ax.set_xlabel('Горизонт, лет');ax.set_ylabel('MAE')
    ax.set_title('Устойчивость к случайной инициализации (фактически выполненные seeds)')
    if len(d):ax.legend(fontsize=7)
figure('04_устойчивость_50_seeds.png',draw_seeds)

# 05 unbiased scatter: only DEV-selected predictors.
def draw_scatter(ax):
    parts=[]
    for r in DEV_WIN.itertuples():
        q=frame_model(r.horizon,r.protocol,r.model,r.feature_set,11)
        if q is not None: parts.append(q[['target_rate','prediction']])
    d=pd.concat(parts,ignore_index=True)
    d=d.sample(min(len(d),3500),random_state=2026)
    ax.hexbin(d.target_rate,d.prediction,gridsize=60,mincnt=1,cmap='viridis',bins='log')
    low=float(min(d.target_rate.min(),d.prediction.min()));high=float(max(d.target_rate.max(),d.prediction.max()))
    ax.plot([low,high],[low,high],'--',lw=1,color='black')
    ax.set_xlabel('Фактический миграционный коэффициент, чел./1000')
    ax.set_ylabel('Прогноз, чел./1000')
    ax.set_title('Факт и прогноз выбранных на DEV моделей (репрезентативная подвыборка)')
figure('05_факт_прогноз.png',draw_scatter)

# 06 temporal errors across all DEV winners without selection by TEST.
def draw_years(ax):
    parts=[]
    for r in DEV_WIN.itertuples():
        q=PRED[(PRED.horizon==r.horizon)&PRED.protocol.eq(r.protocol)&
               PRED.model.eq(r.model)&PRED.feature_set.eq(r.feature_set)&PRED.seed.eq(11)]
        parts.append(q[['horizon','target_year','absolute_error']])
    d=pd.concat(parts,ignore_index=True)
    for h,g in d.groupby('horizon'):
        yr=g.groupby('target_year').absolute_error.mean()
        ax.plot(yr.index,yr.values,marker='o',label=f'{h} год(лет)')
    ax.set_xlabel('Год целевой оценки');ax.set_ylabel('MAE, чел./1000')
    ax.set_title('Изменение ошибки выбранных моделей во времени')
    ax.legend()
figure('06_ошибка_по_годам.png',draw_years)

# 07 coverage; Wilson interval describes iid binomial uncertainty, not spatial dependence.
def draw_interval(ax):
    d=INT_RECALC.sort_values(['horizon','protocol'])
    if d.empty:return
    vals=np.arange(len(d))
    ax.bar(vals,d.coverage,color='#3b7e8a')
    ax.errorbar(vals,d.coverage,yerr=[d.coverage-d.Wilson_low,d.Wilson_high-d.coverage],
                fmt='none',ecolor='black',capsize=2)
    ax.axhline(.9,ls='--',color='#a45b32',label='Номинальные 90%')
    ax.set_xticks(vals,[f'h{int(r.horizon)}\n{r.protocol}' for r in d.itertuples()],rotation=40,ha='right')
    ax.set_ylabel('Доля попаданий в интервал');ax.set_ylim(0,1.08)
    ax.set_title('Эмпирическое покрытие доступных прогнозных интервалов')
    ax.legend()
figure('07_покрытие_интервалов.png',draw_interval)

# 08 calibration with weighted mean (not fitted calibration curve).
def draw_calibration(ax):
    if CALIBRATION.empty:return
    d=CALIBRATION[~CALIBRATION.model.eq('recursive_flow_rule')]
    z=d.groupby(['target','bin'],as_index=False).apply(
        lambda g:pd.Series({'p':np.average(g.mean_probability,weights=g.n),
                            'y':np.average(g.observed_frequency,weights=g.n),
                            'n':g.n.sum()}),include_groups=False)
    for target,g in z.groupby('target'):
        g=g.sort_values('p');ax.plot(g.p,g.y,marker='o',label=target)
    ax.plot([0,1],[0,1],'--',color='black',lw=1)
    ax.set_xlim(0,1);ax.set_ylim(0,1);ax.set_xlabel('Прогнозная вероятность');ax.set_ylabel('Фактическая частота')
    ax.set_title('Калибровка вероятностей (смешанные протоколы — описательно)')
    if len(z):ax.legend(fontsize=8)
figure('08_калибровка_вероятностей.png',draw_calibration)

# 09 Errors by municipality type: use DEV-selected winners only.
def draw_types(ax):
    parts=[]
    for r in DEV_WIN.itertuples():
        q=PRED[(PRED.horizon==r.horizon)&PRED.protocol.eq(r.protocol)&
               PRED.model.eq(r.model)&PRED.feature_set.eq(r.feature_set)&PRED.seed.eq(11)]
        parts.append(q[['mun_type','absolute_error']])
    d=pd.concat(parts,ignore_index=True).groupby('mun_type',as_index=False).agg(MAE=('absolute_error','mean'),n=('absolute_error','size'))
    d=d[d.n>=20].sort_values('MAE').tail(15)
    ax.barh(d.mun_type.astype(str),d.MAE,color='#517f91')
    ax.set_xlabel('MAE, чел./1000')
    ax.set_title('Ошибки по муниципальной типологии (группы с n ≥ 20)')
figure('09_типы_территорий.png',draw_types)

# 10 Distribution of absolute errors (heavy tails).
def draw_distribution(ax):
    parts=[]
    for r in DEV_WIN.itertuples():
        q=PRED[(PRED.horizon==r.horizon)&PRED.protocol.eq(r.protocol)&
               PRED.model.eq(r.model)&PRED.feature_set.eq(r.feature_set)&PRED.seed.eq(11)]
        parts.append(q[['absolute_error']])
    d=pd.concat(parts,ignore_index=True);v=d.absolute_error
    lim=float(v.quantile(.99));ax.hist(v.clip(upper=lim),bins=50,color='#537f87')
    ax.set_xlabel(f'Абсолютная ошибка (99-й перцентиль = {lim:.1f})');ax.set_ylabel('Наблюдений')
    ax.set_title('Распределение ошибок (хвост выше 99-го перцентиля ограничен на графике)')
figure('10_распределение_ошибок.png',draw_distribution)

# A notebook with 10 plots is the only graphics output (ZIP).
FIGURES_ZIP=OUTPUT_DIR/'ML1_V4_PREDICTIONS_AUDIT_FIGURES.zip'
with zipfile.ZipFile(FIGURES_ZIP,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for p in FIGURES:z.write(p,p.name)
with zipfile.ZipFile(FIGURES_ZIP) as z:
    if z.testzip() is not None or len(z.namelist())!=len(FIGURES):
        raise IOError('Не прошла CRC-проверка графиков')
print('Публикационные графики:',len(FIGURES))

# Two Excel only — publication tables and independent QA. No 100s of interim files.
AUDIT_LEVEL='PASS' if not any(x['уровень']=='CRITICAL' for x in QA_ROWS) else 'FAIL'
PROVENANCE=pd.DataFrame([
    {'параметр':'Репозиторий','значение':REPO},
    {'параметр':'GitHub Release','значение':TAG},
    {'параметр':'Коммит','значение':COMMIT},
    {'параметр':'Основной вход','значение':PRED_NAME},
    {'параметр':'SHA256 основного входа','значение':sha256(PRED_FILE)},
    {'параметр':'SHA256 модели/метрик V4','значение':sha256(RESULTS_FILE)},
    {'параметр':'Строк основных прогнозов','значение':len(PRED)},
    {'параметр':'Пар метрик, пересчитанных на TEST','значение':len(RECALC)},
    {'параметр':'Проверка','значение':AUDIT_LEVEL},
    {'параметр':'50 seeds','значение':'ТОЛЬКО отдельные ROLLING_TIME контексты'},
    {'параметр':'ДИ для парных сравнения','значение':'Региональный cluster bootstrap, B='+str(BOOTSTRAPS)},
    {'параметр':'Пост-хок коррекция','значение':'Benjamini–Hochberg для ВСЕХ выполненных абляций'},
    {'параметр':'Предупреждение H3','значение':'Пространственные переменные недоступны без валидного геоключа'},
    {'параметр':'Статус научной интерпретации','значение':'Прогнозные ассоциации; не доказанная причинность'},
    {'параметр':'Дата аудита UTC','значение':datetime.now(timezone.utc).isoformat()},
])
LIMITS=pd.DataFrame([
    {'ограничение':'Исходные данные','следствие':'Ретроспективный срез, оригинальные исторические vintages не проверены'},
    {'ограничение':'Пространственная H3','следствие':'D_spatial не вычисляется без географического ключа'},
    {'ограничение':'DEV-выбор','следствие':'Выбор модели по DEV фиксируется; TEST служит только оценке'},
    {'ограничение':'Повторные наблюдения','следствие':'Региональный bootstrap учитывает региональные блоки, но не полную временную зависимость'},
    {'ограничение':'Множественные проверки','следствие':'Пост-хок p_FDR не превращает гипотезы в пререгистрированные'},
    {'ограничение':'SHAP','следствие':'Объяснимость из опубликованного Model QA; прогнозные корреляции, не причинный эффект'},
    {'ограничение':'Классификация','следствие':'Рекурсивное правило не следует считать калиброванной вероятностной моделью'},
    {'ограничение':'Предсказания','следствие':'Научная проверка выполняется только на опубликованном holdout, без нового обучения'},
])
QA_TABLE=pd.DataFrame(QA_ROWS) if QA_ROWS else pd.DataFrame([{'уровень':'INFO','код':'PASS','описание':'Критических несоответствий не выявлено','число':0}])

BOOKS={
 'ML1_V4_PREDICTIONS_PUBLICATION.xlsx':{
     'Паспорт':PROVENANCE,
     'Метрики_все_модели':RECALC,
     'Победители_DEV':DEV_WIN,
     'Победители_по_факторам':DEV_FAMILY_WIN,
     'Сравнение_с_инерцией':BASELINE_COMP,
     'Абляции_один_алгоритм':ABLATIONS,
     'Устойчивость_seeds':STABILITY,
     'Метрики_по_годам':YEARLY,
     'Метрики_по_регионам':REGIONAL,
     'Метрики_по_типам':TYPES,
     'Абсолютные_потоки':FLOW_RECALC,
     'Классификация':CLASS_RECALC,
     'Калибровка':CALIBRATION,
     'Интервалы':INT_RECALC,
     'Оригинал_H1_H3':PUB_HYPOTHESES,
     'Оригинал_статтесты':PUB_STATTEST,
     'Ограничения':LIMITS,
 },
 'ML1_V4_PREDICTIONS_QA.xlsx':{
     'Паспорт':PROVENANCE,
     'Проверки_качества':QA_TABLE,
     'Состав_листов':SHEET_AUDIT,
     'Контексты_и_истина':CONSISTENCY_BY_CONTEXT,
     'Ключи_покрытие':COVERAGE,
     'Сверка_метрик_V4':METRIC_PARITY,
     'Сверка_потоков':FLOWS_PARITY,
     'Сверка_классификации':CLASS_PARITY,
     'Сверка_интервалов':INTERVAL_PARITY,
     'Ограничения':LIMITS,
 },
}

def safe_cell(v):
    if v is None:return None
    if isinstance(v,(np.integer,)):return int(v)
    if isinstance(v,(np.floating,)):v=float(v)
    if isinstance(v,float) and not np.isfinite(v):return None
    if isinstance(v,(pd.Timestamp,datetime)):return str(v)
    if pd.isna(v):return None
    if isinstance(v,str) and v.lstrip().startswith(('=','+','-','@')):return "'"+v
    return v

def write_xlsx(path,tables):
    tmp=path.with_suffix('.tmp.xlsx')
    wb=xlsxwriter.Workbook(str(tmp),{'constant_memory':True,'strings_to_urls':False,
                                    'strings_to_formulas':False,'nan_inf_to_errors':True})
    head=wb.add_format({'bold':True,'bg_color':'#22435c','font_color':'white','text_wrap':True,
                         'valign':'vcenter','border':0})
    floatfmt=wb.add_format({'num_format':'0.000000'})
    for sheet,df in tables.items():
        if df is None:continue
        df=pd.DataFrame(df)
        if df.shape[0]>1048575 or df.shape[1]>16384:raise ValueError(f'Excel limits exceeded: {sheet}')
        if len(sheet)>31:raise ValueError('Excel sheet name too long '+sheet)
        ws=wb.add_worksheet(sheet);ws.freeze_panes(1,0);ws.set_row(0,32)
        ws.set_column(0,min(df.shape[1]-1,15),19)
        for j,col in enumerate(df.columns):
            ws.write(0,j,str(col),head)
            if j<2:ws.set_column(j,j,24)
        for i,row in enumerate(df.itertuples(index=False,name=None),1):
            for j,value in enumerate(row):
                v=safe_cell(value)
                if isinstance(v,(int,float,np.number)) and not isinstance(v,bool):
                    ws.write_number(i,j,v,floatfmt if isinstance(v,float) else None)
                elif isinstance(v,bool):ws.write_boolean(i,j,v)
                elif v is not None:ws.write_string(i,j,str(v)[:32760])
        if len(df):ws.autofilter(0,0,len(df),df.shape[1]-1)
    wb.close()
    if not zipfile.is_zipfile(tmp):raise IOError('Выходной XLSX не является ZIP: '+str(path))
    tmp.replace(path)
    # independent readback: headers, row counts and ZIP structure.
    check=load_workbook(path,read_only=True,data_only=True)
    try:
        for sheet,df in tables.items():
            if sheet not in check.sheetnames:raise ValueError('Missing result sheet '+sheet)
            ws=check[sheet]
            it=ws.iter_rows(values_only=True)
            cols=[str(x) for x in next(it)]
            if cols!=list(map(str,pd.DataFrame(df).columns)):
                raise ValueError('Header readback failed '+sheet)
            row_count=sum(1 for r in it if any(x is not None for x in r))
            if row_count!=len(df):raise ValueError(f'Row count differs {sheet}: {row_count} vs {len(df)}')
    finally:check.close()
    print('XLSX readback PASS:',path.name,'SHA256',sha256(path),'bytes',path.stat().st_size)

for name,tables in BOOKS.items():write_xlsx(OUTPUT_DIR/name,tables)
print('Готовые файлы:',[p.name for p in sorted(OUTPUT_DIR.iterdir())])

Публикационные графики: 10
XLSX readback PASS: ML1_V4_PREDICTIONS_PUBLICATION.xlsx SHA256 a9b22fb1cae5a0f33e2d86e4930e65faedb17fde6067ade4c76f02057cb68c94 bytes 2149759
XLSX readback PASS: ML1_V4_PREDICTIONS_QA.xlsx SHA256 1f0b0722c899e846db5fe6d5b65bbb3b2257a84cf185c8721ceb33aa972cc538 bytes 73792
Готовые файлы: ['ML1_V4_PREDICTIONS_AUDIT_FIGURES.zip', 'ML1_V4_PREDICTIONS_PUBLICATION.xlsx', 'ML1_V4_PREDICTIONS_QA.xlsx']


## 6. Получение файлов и границы заключения

**Важно:** `PASS` данного пострасчётного аудита относится только к целостности выгрузки, пересчёту статистик и корректности интерпретации. Он **не устраняет** исходные ограничения D01 по историческим vintages, не создаёт пространственные признаки H3, не превращает SHAP в причинные эффекты и не заменяет многолетнюю независимую проверку.

In [10]:
from IPython.display import display, Markdown, FileLinks
assert len(RECALC)==len(PUB_METRICS)
assert len(PRED)==SOURCE_MANIFEST['n_prediction_rows']
assert not any(x['уровень']=='CRITICAL' for x in QA_ROWS)
print('ОКОНЧАТЕЛЬНЫЙ СТАТУС: PASS_REPLAYED_AND_VERIFIED')
print('Основных прогнозов:',len(PRED),'| конфигураций:',len(RECALC),'| независимых сравнений факторов:',len(ABLATIONS))
print('Классификаций:',len(CLASS_RECALC),'| интервальных контекстов:',len(INT_RECALC),'| графиков:',len(FIGURES))
print('Региональные bootstrap интервалы: B =',BOOTSTRAPS)
display(FileLinks(str(OUTPUT_DIR)))
try:
    from google.colab import files
    print('Для скачивания отдельных результатов доступны три ссылки выше; автоотправка файлов не включена.')
except ImportError:pass

ОКОНЧАТЕЛЬНЫЙ СТАТУС: PASS_REPLAYED_AND_VERIFIED
Основных прогнозов: 870660 | конфигураций: 486 | независимых сравнений факторов: 288
Классификаций: 48 | интервальных контекстов: 4 | графиков: 10
Региональные bootstrap интервалы: B = 1500


/content/ML1_V4_PREDICTION_AUDIT/publication/
  ML1_V4_PREDICTIONS_QA.xlsx
  ML1_V4_PREDICTIONS_PUBLICATION.xlsx
  ML1_V4_PREDICTIONS_AUDIT_FIGURES.zip

Для скачивания отдельных результатов доступны три ссылки выше; автоотправка файлов не включена.
